# GoEmotions data exploration (P1)

Run every cell from top to bottom. This notebook summarizes the supplied raw and simplified files; it does not train a model or test a hypothesis. Counts and descriptions are generated from the local data, not entered by hand.

In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / "data"

EMOTIONS = (DATA_DIR / "simplified" / "emotions.txt").read_text(encoding="utf-8").splitlines()
RAW_FILES = sorted((DATA_DIR / "raw").glob("goemotions_*.csv"))
SPLIT_FILES = {name: DATA_DIR / "simplified" / f"{name}.tsv" for name in ("train", "dev", "test")}

assert len(RAW_FILES) == 3, f"Expected 3 raw CSV files; found {len(RAW_FILES)}"
assert all(path.is_file() for path in SPLIT_FILES.values()), "A simplified split file is missing"

raw_parts = [pd.read_csv(path, low_memory=False) for path in RAW_FILES]
assert all(part.columns.equals(raw_parts[0].columns) for part in raw_parts[1:]), "Raw CSV schemas differ"
raw = pd.concat(raw_parts, ignore_index=True)
splits = {
    name: pd.read_csv(path, sep="\t", header=None, names=["text", "emotion_ids", "id"], dtype="string")
    for name, path in SPLIT_FILES.items()
}
raw_comments = raw.drop_duplicates("id").copy()
split_ids = {name: set(frame["id"].dropna()) for name, frame in splits.items()}
all_labeled_ids = set.union(*split_ids.values())

print(f"Raw files: {len(RAW_FILES)}; raw rows: {len(raw):,}; unique comment IDs: {raw['id'].nunique():,}")
print(f"Emotion labels: {len(EMOTIONS)}")

Raw files: 3; raw rows: 211,225; unique comment IDs: 58,011
Emotion labels: 28


## 1. Row counts and split sizes

In [2]:
pd.DataFrame({"rows": {name: len(frame) for name, frame in splits.items()}}).assign(
    unique_ids={name: frame["id"].nunique() for name, frame in splits.items()}
)

,rows,unique_ids
train,43410,43410
dev,5426,5426
test,5427,5427


## 2. Unique comments and raters per comment

In [3]:
raters_per_comment = raw.groupby("id")["rater_id"].nunique()
raters_per_comment.describe(percentiles=[0.25, 0.5, 0.75, 0.9]).to_frame("raters_per_comment")

,raters_per_comment
count,58011.000000
mean,3.641120
std,0.935532
min,1.000000
25%,3.000000
50%,3.000000
75%,5.000000
90%,5.000000
max,5.000000


## 3. Columns and missing values

In [4]:
pd.DataFrame({"dtype": raw.dtypes.astype(str), "missing": raw.isna().sum()}).sort_values("missing", ascending=False)

,dtype,missing
text,str,0
id,str,0
author,str,0
subreddit,str,0
link_id,str,0
parent_id,str,0
created_utc,float64,0
rater_id,int64,0
example_very_unclear,bool,0
admiration,int64,0


## 4–5. Most common subreddits and comment-length distribution

Subreddit counts use one row per comment. Length is the number of whitespace-separated words.

In [5]:
top_subreddits = raw_comments["subreddit"].value_counts().head(15).rename_axis("subreddit").to_frame("comments")
word_count = raw_comments["text"].fillna("").str.split().str.len()
length_summary = word_count.describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95]).to_frame("word_count")
display(top_subreddits)
display(length_summary)

,comments
subreddit,
loveafterlockup,241
cringe,239
socialanxiety,232
AnimalsBeingBros,231
vanderpumprules,221
90dayfianceuncensored,221
confessions,220
timberwolves,220
TheSimpsons,219


,word_count
count,58011.000000
mean,12.913758
std,6.688405
min,1.000000
25%,7.000000
50%,12.000000
75%,18.000000
90%,23.000000
95%,24.000000
max,33.000000


## 6–7. Label counts and multi-label share

In [6]:
def parse_label_ids(value):
    if pd.isna(value) or not str(value).strip():
        return []
    return [int(item) for item in str(value).split(",")]

label_lists = {name: frame["emotion_ids"].map(parse_label_ids) for name, frame in splits.items()}
label_counts = pd.Series(
    [label_id for labels in label_lists.values() for row in labels for label_id in row]
).value_counts().reindex(range(len(EMOTIONS)), fill_value=0).sort_index()
label_counts.index = EMOTIONS
display(label_counts.rename("positive labels").to_frame())

all_label_rows = [labels for split_labels in label_lists.values() for labels in split_labels]
multi_label_share = sum(len(labels) > 1 for labels in all_label_rows) / len(all_label_rows)
print(f"Rows with more than one label: {multi_label_share:.1%}")

,positive labels
admiration,5122
amusement,2895
anger,1960
annoyance,3093
approval,3687
caring,1375
confusion,1673
curiosity,2723
desire,801
disappointment,1583


Rows with more than one label: 16.2%


## 8. Comments present in raw data but absent from simplified splits

In [7]:
print(f"Unique simplified IDs: {len(all_labeled_ids):,}")
print(f"Raw IDs not in any simplified split: {len(set(raw_comments['id']) - all_labeled_ids):,}")

split_overlap = {
    f"{left}/{right}": len(split_ids[left] & split_ids[right])
    for left, right in (("train", "dev"), ("train", "test"), ("dev", "test"))
}
print("IDs shared across splits:", split_overlap)

Unique simplified IDs: 54,263
Raw IDs not in any simplified split: 3,748
IDs shared across splits: {'train/dev': 0, 'train/test': 0, 'dev/test': 0}


## 9. `example_very_unclear` flags

In [8]:
unclear = raw["example_very_unclear"].astype("string").str.lower().isin(["true", "1", "yes"])
unclear_by_comment = raw.assign(_unclear=unclear).groupby("id")["_unclear"].any()
print(f"Flagged raw rater rows: {int(unclear.sum()):,}")
print(f"Unique comments flagged at least once: {int(unclear_by_comment.sum()):,}")

Flagged raw rater rows: 3,411
Unique comments flagged at least once: 2,733


## 10. Consistency checks

Resolve unexpected split sizes or overlapping IDs before modeling; document the outcome in `docs/log.md`.

In [9]:
expected_rows = {"train": 43_410, "dev": 5_426, "test": 5_427}
actual_rows = {name: len(frame) for name, frame in splits.items()}
print("Expected split sizes:", expected_rows)
print("Actual split sizes:  ", actual_rows)
assert actual_rows == expected_rows, "Simplified split sizes differ from the whiteboard"
assert not any(split_overlap.values()), f"Comment IDs overlap across splits: {split_overlap}"

emotion_columns = set(EMOTIONS)
assert emotion_columns.issubset(raw.columns), "One or more emotion columns are missing from the raw CSVs"
print("Split sizes and non-overlap checks passed.")

Expected split sizes: {'train': 43410, 'dev': 5426, 'test': 5427}
Actual split sizes:   {'train': 43410, 'dev': 5426, 'test': 5427}
Split sizes and non-overlap checks passed.


## GoEmotions paper notes — researcher to complete

Read Demszky et al. (2020), *GoEmotions: A Dataset of Fine-Grained Emotions*, and write about one page in your own words. Address: (1) why and how the dataset was collected, (2) its emotion taxonomy and labeling process, (3) the authors' reported limitations, (4) which details matter for this project, and (5) what you still need to verify in the full paper. Do not treat this prompt as a substitute for reading the paper.